In [4]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np

from src.utils.delaunay2d import *
from collections import defaultdict 

import json
from pathlib import Path
from src.formatResults import *
import pandas as pd
import warnings

import matplotlib.pyplot as plt
from src.smithMethods import *

from synthethicExperimentsUtils import *
    
warnings.filterwarnings("ignore", category=RuntimeWarning) 

FIG_PATH =  Path("./Figures/Results")
RESULT_PATH =  Path("./Results")

from src.exhaustiveMethods_local import vanillaMST
from src.embed.distances import *
from src.embed.tree_embedders import *

def seed_all(seed=42):
    np.random.seed(seed)
    random.seed(seed)

PHCv2.4.86 released 2022-05-30 works!


# Experiments

## Exp 1: Centered Gaussian

In [3]:
# NJ
if True:
    numeric_results = defaultdict(lambda: list())
    samplingParam= {"cov": np.eye(2)*0.5}
    nIters = 3
    convDiff = 1e-1
    dist2Points = 1e-5
    numSamples = 2
    title = "CenteredGauss"
    method="NJ"

    space = "Klein"
    sampleType = "WrappedGauss"
     
    
    # "Euclidean", "Klein_3+Precise", "Klein_3+Simple", "Klein_4+Simple"
    for mode in ["NJ"]:
            
        for numPoints in [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200]:

            seed_all(10)

            avgTimeList = []
            ratioList = []
            perImprovList = []

            MSTvalList = []

            i=0
            while i < numSamples:
                
            
                print(f"\rIter: {i}", end="")
                    
                points_klein = sampleExperiments(sampleType, numPoints, samplingParam=samplingParam)
                vertices_dict_terminal = {f"T{j}": points_klein[j] for j in range(numPoints)}
                mstEdgeList, mstVert2Adj, MSTval = vanillaMST(vertices_dict_terminal, space=space)

                # Neighbor-Joining
                start_time_nj = time.process_time()
                distance_matrix_points_klein = distance_matrix(points_klein)
                njGraph, nj_networkx = neighbor_joining(points_klein, distance_matrix_points_klein) 
                del distance_matrix_points_klein ###
                nj_adjacency_matrix, nj_nodes_ordered = adjacency_matrix(nj_networkx) 

                nj_adjacency_matrix = torch.tensor(nj_adjacency_matrix)
                poincare = klein_to_poincare(points_klein)
                poincare = torch.tensor(poincare)

                nj_steiners = train_steiner_embeddings(
                    nj_adjacency_matrix,
                    terminals_poincare=poincare,
                    num_epochs=10000,
                    lr=1)

                combined = torch.cat([poincare, nj_steiners], dim=0)
                combined_klein = poincare_to_klein(combined)
                dist_matrix = distance_matrix(combined_klein, klein_distance)        
                nj_length = (0.5*(dist_matrix*nj_adjacency_matrix)).sum()
                nj_time = time.process_time() - start_time_nj

                # Convert tensor to numpy value and clean up memory
                nj_length_value = nj_length.detach().cpu().numpy().item()
                
                # Clean up tensors to free memory
                del nj_adjacency_matrix, poincare, nj_steiners, combined, combined_klein, dist_matrix, nj_length
                
                #   # Force garbage collection to free GPU/CPU memory
                #   if torch.cuda.is_available():
                #       torch.cuda.empty_cache()
                #   gc.collect()

                avgTimeList.append(nj_time)
                ratioList.append(nj_length_value/MSTval)
                perImprovList.append((MSTval-nj_length_value)/MSTval)

                MSTvalList.append(MSTval)
                i += 1



            avgTime = np.mean(avgTimeList)
            avgTime_std = np.std(avgTimeList)
            ratio = np.mean(ratioList)
            ratio_std = np.std(ratioList)
            perImprov = np.mean(perImprovList)*100
            perImprov_std = np.std(perImprovList)*100


            print(f"\r{mode}, {numPoints}, {avgTime}+-{avgTime_std}, {perImprov}+-{perImprov_std}, {ratio}+-{ratio_std}")

            numeric_results["mode"] += [mode]*numSamples
            numeric_results["numPoints"] += [numPoints]*numSamples
            numeric_results["avgTime"] += avgTimeList#.tolist()
            numeric_results["ratio"] += ratioList#.tolist()
            numeric_results["perImprov"] += perImprovList#.tolist()           
            

        print("-"*30)

    
    df = pd.DataFrame(numeric_results)


    df.to_csv(
        RESULT_PATH / f'results_{title}_{method}.tsv',
        sep="\t",
        index=False
        )

In [ ]:
# HS
if True:
    numeric_results = defaultdict(lambda: list())
    samplingParam= {"cov": np.eye(2)*0.5}
    nIters = 3
    convDiff = 1e-1
    dist2Points = 1e-5
    numSamples = 10
    title = "CenteredGauss"
    method="SLL"
    
    for mode in ["Klein_4+Simple"]:
        
       
        
        if mode == "Euclidean":
            sampleType = "EucGauss"
            fstSize = 4
            space = "Euclidean"
            precise=True
        else:
            sampleType = "WrappedGauss"
            space = "Klein"
            fstSize = int(mode[6])
            precise = mode[8:] == "Precise"
                
        for numPoints in [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200]:
            
            nMaxExpansions=int(1*np.sqrt(numPoints))
            resultExp = syntheticExperiments(method=method,
                                                numSamples=numSamples, 
                                                numPoints=numPoints,
                                                nIters=nIters,
                                                fstSize=fstSize,
                                                convDiff=convDiff,
                                                dist2Points=dist2Points,
                                                precise=precise, 
                                                sampleType=sampleType,
                                                samplingParam=samplingParam,
                                                space=space,
                                                nMaxExpansions=nMaxExpansions,
                                                seed=10,
                                                selection= ("MST", (1, 1)))




            correctList, errVorList, avgTimeList, ratioList, perImprovList, avgFST3List, avgFST4List = resultExp

            correct = np.mean(correctList)*100
            avgTime = np.mean(avgTimeList)
            avgTime_std = np.std(avgTimeList)
            ratio = np.mean(ratioList)
            ratio_std = np.std(ratioList)
            perImprov = np.mean(perImprovList)*100
            perImprov_std = np.std(perImprovList)*100
            avgFST3 = np.mean(avgFST3List)
            avgFST4 = np.mean(avgFST4List)

            print(f"\r{mode}, {numPoints}, {avgTime}+-{avgTime_std}, {perImprov}+-{perImprov_std}, {ratio}+-{ratio_std}")

            numeric_results["mode"] += [mode]*numSamples
            numeric_results["numPoints"] += [numPoints]*numSamples
            numeric_results["correct"] += correctList.tolist()
            numeric_results["avgTime"] += avgTimeList.tolist()
            numeric_results["ratio"] += ratioList.tolist()
            numeric_results["perImprov"] += perImprovList.tolist()
            numeric_results["avgFST3"] += avgFST3List.tolist()
            numeric_results["avgFST4"] += avgFST4List.tolist()

           
            

        print("-"*30)

    
    df = pd.DataFrame(numeric_results)


    df.to_csv(
        RESULT_PATH / f'results_{title}_{method}.tsv',
        sep="\t",
        index=False
        )

In [2]:
# RHS
if True:
    numeric_results = defaultdict(lambda: list())
    samplingParam= {"cov": np.eye(2)*0.5}
    nIters = 3
    convDiff = 1e-1
    dist2Points = 1e-5
    numSamples = 10
    title = "CenteredGauss"
    method="EXH"
    
    for mode in ["Klein_4+Simple"]:
        
       
        
        if mode == "Euclidean":
            sampleType = "EucGauss"
            fstSize = 4
            space = "Euclidean"
            precise=True
        else:
            sampleType = "WrappedGauss"
            space = "Klein"
            fstSize = int(mode[6])
            precise = mode[8:] == "Precise"
                
        for numPoints in [20, 50, 60, 70, 80, 90, 100, 200]:
            
            nMaxExpansions=int(1*np.sqrt(numPoints))
            resultExp = syntheticExperiments(method=method,
                                                numSamples=numSamples, 
                                                numPoints=numPoints,
                                                nIters=nIters,
                                                fstSize=fstSize,
                                                convDiff=convDiff,
                                                dist2Points=dist2Points,
                                                precise=precise, 
                                                sampleType=sampleType,
                                                samplingParam=samplingParam,
                                                space=space,
                                                nMaxExpansions=nMaxExpansions,
                                                seed=10,
                                                slack=2,
                                                selection=("Prob", (0.3, 0.6)),
                                                threshold_improvement=0.0,
                                                num_epochs=10000,
                                                lr=1e-2,
                                                early_stopping=True,
                                                patience=100,
                                                min_delta=1e-6)




            correctList, errVorList, avgTimeList, ratioList, perImprovList, avgFST3List, avgFST4List = resultExp

            correct = np.mean(correctList)*100
            avgTime = np.mean(avgTimeList)
            avgTime_std = np.std(avgTimeList)
            ratio = np.mean(ratioList)
            ratio_std = np.std(ratioList)
            perImprov = np.mean(perImprovList)*100
            perImprov_std = np.std(perImprovList)*100
            avgFST3 = np.mean(avgFST3List)
            avgFST4 = np.mean(avgFST4List)

            print(f"\r{mode}, {numPoints}, {avgTime}+-{avgTime_std}, {perImprov}+-{perImprov_std}, {ratio}+-{ratio_std}")

            numeric_results["mode"] += [mode]*numSamples
            numeric_results["numPoints"] += [numPoints]*numSamples
            numeric_results["correct"] += correctList.tolist()
            numeric_results["avgTime"] += avgTimeList.tolist()
            numeric_results["ratio"] += ratioList.tolist()
            numeric_results["perImprov"] += perImprovList.tolist()
            numeric_results["avgFST3"] += avgFST3List.tolist()
            numeric_results["avgFST4"] += avgFST4List.tolist()

           
            

        print("-"*30)

    
    df = pd.DataFrame(numeric_results)


    df.to_csv(
        RESULT_PATH / f'results_{title}_{method}.tsv',
        sep="\t",
        index=False
        )

## Exp 2: Mixture of 10 Gaussians

In [ ]:
# HS exp2
if True:
    numeric_results = defaultdict(lambda: list())
    sampleType = "PolyWrappedGauss"
    samplingParam= {"cov": np.eye(2)*0.1, "numPoly": 10, "t": 1-1e-10}
    space = "Klein"
    nIters = 3
    convDiff = 1e-1
    dist2Points = 1e-5
    numSamples = 10
    title = "PolyWrappedGauss"
    method="SLL"


    for mode in ["Klein_4+Simple"]:
        
       
        fstSize = int(mode[6])
        precise = mode[8:] == "Precise"
                
        for numPoints in [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200]:
            
            nMaxExpansions=int(1*np.sqrt(numPoints))
            resultExp = syntheticExperiments(method=method,
                                                numSamples=numSamples, 
                                                numPoints=numPoints,
                                                nIters=nIters,
                                                fstSize=fstSize,
                                                convDiff=convDiff,
                                                dist2Points=dist2Points,
                                                precise=precise, 
                                                sampleType=sampleType,
                                                samplingParam=samplingParam,
                                                space=space,
                                                nMaxExpansions=nMaxExpansions,
                                                seed=10,
                                                selection= ("MST", (1, 1)))




            correctList, errVorList, avgTimeList, ratioList, perImprovList, avgFST3List, avgFST4List = resultExp

            correct = np.mean(correctList)*100
            avgTime = np.mean(avgTimeList)
            avgTime_std = np.std(avgTimeList)
            ratio = np.mean(ratioList)
            ratio_std = np.std(ratioList)
            perImprov = np.mean(perImprovList)*100
            perImprov_std = np.std(perImprovList)*100
            avgFST3 = np.mean(avgFST3List)
            avgFST4 = np.mean(avgFST4List)

            print(f"\r{mode}, {numPoints}, {avgTime}+-{avgTime_std}, {perImprov}+-{perImprov_std}, {ratio}+-{ratio_std}")

            numeric_results["mode"] += [mode]*numSamples
            numeric_results["numPoints"] += [numPoints]*numSamples
            numeric_results["correct"] += correctList.tolist()
            numeric_results["avgTime"] += avgTimeList.tolist()
            numeric_results["ratio"] += ratioList.tolist()
            numeric_results["perImprov"] += perImprovList.tolist()
            numeric_results["avgFST3"] += avgFST3List.tolist()
            numeric_results["avgFST4"] += avgFST4List.tolist()

           
            

        print("-"*30)

    
    df = pd.DataFrame(numeric_results)


    df.to_csv(
        RESULT_PATH / f'results_{title}_{method}.tsv',
        sep="\t",
        index=False
        )

In [4]:
# RHS exp2
if True:
    numeric_results = defaultdict(lambda: list())
    sampleType = "PolyWrappedGauss"
    samplingParam= {"cov": np.eye(2)*0.1, "numPoly": 10, "t": 1-1e-10}
    space = "Klein"
    nIters = 3
    convDiff = 1e-1
    dist2Points = 1e-5
    numSamples = 10
    title = "PolyWrappedGauss"
    method="EXH"
    
    for mode in ["Klein_4+Simple"]:
        
        fstSize = int(mode[6])
        precise = mode[8:] == "Precise"
                
        for numPoints in [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200]:
            
            nMaxExpansions=int(1*np.sqrt(numPoints))
            resultExp = syntheticExperiments(method=method,
                                                numSamples=numSamples, 
                                                numPoints=numPoints,
                                                nIters=nIters,
                                                fstSize=fstSize,
                                                convDiff=convDiff,
                                                dist2Points=dist2Points,
                                                precise=precise, 
                                                sampleType=sampleType,
                                                samplingParam=samplingParam,
                                                space=space,
                                                nMaxExpansions=nMaxExpansions,
                                                seed=10,
                                                slack=2,
                                                selection=("Prob", (0.3, 0.6)),
                                                threshold_improvement=0.0,
                                                num_epochs=10000,
                                                lr=1e-2,
                                                early_stopping=True,
                                                patience=100,
                                                min_delta=1e-6)




            correctList, errVorList, avgTimeList, ratioList, perImprovList, avgFST3List, avgFST4List = resultExp

            correct = np.mean(correctList)*100
            avgTime = np.mean(avgTimeList)
            avgTime_std = np.std(avgTimeList)
            ratio = np.mean(ratioList)
            ratio_std = np.std(ratioList)
            perImprov = np.mean(perImprovList)*100
            perImprov_std = np.std(perImprovList)*100
            avgFST3 = np.mean(avgFST3List)
            avgFST4 = np.mean(avgFST4List)

            print(f"\r{mode}, {numPoints}, {avgTime}+-{avgTime_std}, {perImprov}+-{perImprov_std}, {ratio}+-{ratio_std}")

            numeric_results["mode"] += [mode]*numSamples
            numeric_results["numPoints"] += [numPoints]*numSamples
            numeric_results["correct"] += correctList.tolist()
            numeric_results["avgTime"] += avgTimeList.tolist()
            numeric_results["ratio"] += ratioList.tolist()
            numeric_results["perImprov"] += perImprovList.tolist()
            numeric_results["avgFST3"] += avgFST3List.tolist()
            numeric_results["avgFST4"] += avgFST4List.tolist()

           
            

        print("-"*30)

    
    df = pd.DataFrame(numeric_results)


    df.to_csv(
        RESULT_PATH / f'results_{title}_{method}.tsv',
        sep="\t",
        index=False
        )

In [ ]:
# NJ exp2
if True:
    numeric_results = defaultdict(lambda: list())
    sampleType = "PolyWrappedGauss"
    samplingParam= {"cov": np.eye(2)*0.1, "numPoly": 10, "t": 1-1e-10}
    space = "Klein"
    nIters = 3
    convDiff = 1e-1
    dist2Points = 1e-5
    numSamples = 2
    title = "PolyWrappedGauss"
    method="NJ"

    
    for mode in ["NJ"]:
            
        for numPoints in [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200]:

            seed_all(10)

            avgTimeList = []
            ratioList = []
            perImprovList = []

            MSTvalList = []

            i=0
            while i < numSamples:
                
            
                print(f"\rIter: {i}", end="")
                    
                points_klein = sampleExperiments(sampleType, numPoints, samplingParam=samplingParam)
                vertices_dict_terminal = {f"T{j}": points_klein[j] for j in range(numPoints)}
                mstEdgeList, mstVert2Adj, MSTval = vanillaMST(vertices_dict_terminal, space=space)

                # Neighbor-Joining
                start_time_nj = time.process_time()
                distance_matrix_points_klein = distance_matrix(points_klein)
                njGraph, nj_networkx = neighbor_joining(points_klein, distance_matrix_points_klein) 
                del distance_matrix_points_klein 
                nj_adjacency_matrix, nj_nodes_ordered = adjacency_matrix(nj_networkx) 

                nj_adjacency_matrix = torch.tensor(nj_adjacency_matrix)
                poincare = klein_to_poincare(points_klein)
                poincare = torch.tensor(poincare)

                nj_steiners = train_steiner_embeddings(
                    nj_adjacency_matrix,
                    terminals_poincare=poincare,
                    num_epochs=10000,
                    lr=1)

                combined = torch.cat([poincare, nj_steiners], dim=0)
                combined_klein = poincare_to_klein(combined)
                dist_matrix = distance_matrix(combined_klein, klein_distance)        
                nj_length = (0.5*(dist_matrix*nj_adjacency_matrix)).sum()
                nj_time = time.process_time() - start_time_nj

                # Convert tensor to numpy value and clean up memory
                nj_length_value = nj_length.detach().cpu().numpy().item()
                
                # Clean up tensors to free memory
                del nj_adjacency_matrix, poincare, nj_steiners, combined, combined_klein, dist_matrix, nj_length
                
                #   # Force garbage collection to free GPU/CPU memory
                #   if torch.cuda.is_available():
                #       torch.cuda.empty_cache()
                #   gc.collect()

                avgTimeList.append(nj_time)
                ratioList.append(nj_length_value/MSTval)
                perImprovList.append((MSTval-nj_length_value)/MSTval)

                MSTvalList.append(MSTval)
                i += 1



            avgTime = np.mean(avgTimeList)
            avgTime_std = np.std(avgTimeList)
            ratio = np.mean(ratioList)
            ratio_std = np.std(ratioList)
            perImprov = np.mean(perImprovList)*100
            perImprov_std = np.std(perImprovList)*100


            print(f"\r{mode}, {numPoints}, {avgTime}+-{avgTime_std}, {perImprov}+-{perImprov_std}, {ratio}+-{ratio_std}")

            numeric_results["mode"] += [mode]*numSamples
            numeric_results["numPoints"] += [numPoints]*numSamples
            numeric_results["avgTime"] += avgTimeList#.tolist()
            numeric_results["ratio"] += ratioList#.tolist()
            numeric_results["perImprov"] += perImprovList#.tolist()           
            

        print("-"*30)

    
    df = pd.DataFrame(numeric_results)


    df.to_csv(
        RESULT_PATH / f'results_{title}_{method}.tsv',
        sep="\t",
        index=False
        )

## Exp 3: Approaching the Theoretical Limit

In [ ]:
# HS exp3
if True:
    numeric_results = defaultdict(lambda: list())
    sampleType = "PolyWrappedGauss" 
    space = "Klein"
    nIters = 3
    convDiff = 1e-1
    dist2Points = 1e-5
    numSamples = 3
    title = "Convergence"
    method="SLL"
    
    for mode in ["Klein_4+Simple"]:
        
        fstSize = int(mode[6])
        precise = mode[8:] == "Precise"
                
        for numPoints in [4,5,6,7,8,9,10, 20, 30, 40, 50, 60,70,80,90,100,200]:
            samplingParam= {"cov": np.eye(2)*0.1, "numPoly": numPoints, "t": 1-1e-10}
            
            nMaxExpansions=int(1*np.sqrt(numPoints))

            max_retries = 1000  # Maximum number of different seeds to try
            base_seed = 0
            resultExp = None

            for retry in range(max_retries):
                current_seed = base_seed + retry
                try:          
                    resultExp = syntheticExperiments(method=method,
                                                        numSamples=numSamples, 
                                                        numPoints=numPoints,
                                                        nIters=nIters,
                                                        fstSize=fstSize,
                                                        convDiff=convDiff,
                                                        dist2Points=dist2Points,
                                                        precise=precise, 
                                                        sampleType=sampleType,
                                                        samplingParam=samplingParam,
                                                        space=space,
                                                        nMaxExpansions=nMaxExpansions,
                                                        seed=current_seed,
                                                        selection= ("MST", (1, 1)))
                            
                    correctList, errVorList, avgTimeList, ratioList, perImprovList, avgFST3List, avgFST4List = resultExp
                    
                    while (min(perImprovList)<=0):
                        current_seed+=1
                        resultExp = syntheticExperiments(method=method,
                                                            numSamples=numSamples, 
                                                            numPoints=numPoints,
                                                            nIters=nIters,
                                                            fstSize=fstSize,
                                                            convDiff=convDiff,
                                                            dist2Points=dist2Points,
                                                            precise=precise, 
                                                            sampleType=sampleType,
                                                            samplingParam=samplingParam,
                                                            space=space,
                                                            nMaxExpansions=nMaxExpansions,
                                                            seed=current_seed,
                                                            selection= ("MST", (1, 1)))       
                        correctList, errVorList, avgTimeList, ratioList, perImprovList, avgFST3List, avgFST4List = resultExp
                        #print(perImprovList)                    

                    print(f"Success with seed {current_seed} for numPoints={numPoints}")
                    break
                except Exception as e:
                    print(f"Error with seed {current_seed} for numPoints={numPoints}: {str(e)}")
                    if retry == max_retries - 1:
                        raise e 
                    continue


            correct = np.mean(correctList)*100
            avgTime = np.mean(avgTimeList)
            avgTime_std = np.std(avgTimeList)
            print(perImprovList)
            ratio = np.mean(ratioList)
            ratio_std = np.std(ratioList)
            perImprov = np.mean(perImprovList)*100
            perImprov_std = np.std(perImprovList)*100
            avgFST3 = np.mean(avgFST3List)
            avgFST4 = np.mean(avgFST4List)

            best_ratio = np.max(ratioList)
            best_perImprov = np.max(perImprovList)*100


            print(f"\r{numPoints}, {avgTime}+-{avgTime_std}, {best_perImprov}, {perImprov}+-{perImprov_std}, {ratio}+-{ratio_std}")

            numeric_results["mode"] += [mode]*numSamples
            numeric_results["numPoints"] += [numPoints]*numSamples
            numeric_results["correct"] += correctList.tolist()
            numeric_results["avgTime"] += avgTimeList.tolist()
            numeric_results["ratio"] += ratioList.tolist()
            numeric_results["perImprov"] += perImprovList.tolist()
            numeric_results["avgFST3"] += avgFST3List.tolist()
            numeric_results["avgFST4"] += avgFST4List.tolist()

    
            df = pd.DataFrame(numeric_results)


            df.to_csv(
                RESULT_PATH / f'results_{title}_{method}.tsv',
                sep="\t",
                index=False
                )

        print("-"*30)

In [ ]:
# RHS exp3 
if True:
    numeric_results = defaultdict(lambda: list())
    sampleType = "PolyWrappedGauss" 
    space = "Klein"
    nIters = 3
    convDiff = 1e-1
    dist2Points = 1e-5
    numSamples = 3
    title = "Convergence"
    method="EXH"
    
    for mode in ["Klein_4+Simple"]:
        
        fstSize = int(mode[6])
        precise = mode[8:] == "Precise"
                
        for numPoints in [4,5,6,7,8,9,10, 20, 30, 40, 50, 60,70,80,90,100,200]:
            samplingParam= {"cov": np.eye(2)*0.1, "numPoly": numPoints, "t": 1-1e-10}
            
            nMaxExpansions=int(1*np.sqrt(numPoints))

            max_retries = 1000  # Maximum number of different seeds to try
            base_seed = 0
            resultExp = None

            for retry in range(max_retries):
                current_seed = base_seed + retry
                try:          
                    resultExp = syntheticExperiments(method=method,
                                                        numSamples=numSamples, 
                                                        numPoints=numPoints,
                                                        nIters=nIters,
                                                        fstSize=fstSize,
                                                        convDiff=convDiff,
                                                        dist2Points=dist2Points,
                                                        precise=precise, 
                                                        sampleType=sampleType,
                                                        samplingParam=samplingParam,
                                                        space=space,
                                                        nMaxExpansions=nMaxExpansions,
                                                        seed=current_seed,
                                                        slack=2,
                                                        selection=("Prob", (0.3, 0.6)),
                                                        threshold_improvement=0.0,
                                                        num_epochs=10000,
                                                        lr=1e-2,
                                                        early_stopping=True,
                                                        patience=100,
                                                        min_delta=1e-6)
                    
                    correctList, errVorList, avgTimeList, ratioList, perImprovList, avgFST3List, avgFST4List = resultExp
                    while (min(perImprovList)<=0):
                        current_seed+=1
                        resultExp = syntheticExperiments(method=method,
                                                            numSamples=numSamples, 
                                                            numPoints=numPoints,
                                                            nIters=nIters,
                                                            fstSize=fstSize,
                                                            convDiff=convDiff,
                                                            dist2Points=dist2Points,
                                                            precise=precise, 
                                                            sampleType=sampleType,
                                                            samplingParam=samplingParam,
                                                            space=space,
                                                            nMaxExpansions=nMaxExpansions,
                                                            seed=current_seed,
                                                            slack=2,
                                                            selection=("Prob", (0.3, 0.6)),
                                                            threshold_improvement=0.0,
                                                            num_epochs=10000,
                                                            lr=1e-2,
                                                            early_stopping=True,
                                                            patience=100,
                                                            min_delta=1e-6)        
                        correctList, errVorList, avgTimeList, ratioList, perImprovList, avgFST3List, avgFST4List = resultExp

                    

                    print(f"Success with seed {current_seed} for numPoints={numPoints}")
                    break
                except Exception as e:
                    print(f"Error with seed {current_seed} for numPoints={numPoints}: {str(e)}")
                    if retry == max_retries - 1:
                        raise e  
                    continue


            correct = np.mean(correctList)*100
            avgTime = np.mean(avgTimeList)
            avgTime_std = np.std(avgTimeList)
            print(perImprovList)
            ratio = np.mean(ratioList)
            ratio_std = np.std(ratioList)
            perImprov = np.mean(perImprovList)*100
            perImprov_std = np.std(perImprovList)*100
            avgFST3 = np.mean(avgFST3List)
            avgFST4 = np.mean(avgFST4List)

            best_ratio = np.max(ratioList)
            best_perImprov = np.max(perImprovList)*100


            print(f"\r{numPoints}, {avgTime}+-{avgTime_std}, {best_perImprov}, {perImprov}+-{perImprov_std}, {ratio}+-{ratio_std}")

            numeric_results["mode"] += [mode]*numSamples
            numeric_results["numPoints"] += [numPoints]*numSamples
            numeric_results["correct"] += correctList.tolist()
            numeric_results["avgTime"] += avgTimeList.tolist()
            numeric_results["ratio"] += ratioList.tolist()
            numeric_results["perImprov"] += perImprovList.tolist()
            numeric_results["avgFST3"] += avgFST3List.tolist()
            numeric_results["avgFST4"] += avgFST4List.tolist()

    
            df = pd.DataFrame(numeric_results)


            df.to_csv(
                RESULT_PATH / f'results_{title}_{method}.tsv',
                sep="\t",
                index=False
                )

        print("-"*30)

In [ ]:
# NJ exp3 
if True:
    numeric_results = defaultdict(lambda: list())
    sampleType = "PolyWrappedGauss" 
    space = "Klein"
    nIters = 3
    convDiff = 1e-1
    dist2Points = 1e-5
    numSamples = 3
    title = "Convergence"
    method="NJ"
    
    for mode in ["NJ"]:
        
        for numPoints in [4,5,6,7,8,9,10, 20, 30, 40, 50, 60,70,80,90,100,200]:
            samplingParam= {"cov": np.eye(2)*0.1, "numPoly": numPoints, "t": 1-1e-10}
            
            nMaxExpansions=int(1*np.sqrt(numPoints))

            max_retries = 1000  # Maximum number of different seeds to try
            current_seed = 0
            resultExp = None

            avgTimeList = []
            ratioList = []
            perImprovList = []

            MSTvalList = []

            i = 0
            while i < numSamples:

                for retry in range(max_retries):
                    current_seed = current_seed + retry
                    try:        

                        seed_all(current_seed)

                        points_klein = sampleExperiments(sampleType, numPoints, samplingParam=samplingParam)
                        vertices_dict_terminal = {f"T{j}": points_klein[j] for j in range(numPoints)}
                        mstEdgeList, mstVert2Adj, MSTval = vanillaMST(vertices_dict_terminal, space=space)


                        # Neighbor-Joining

                        start_time_nj = time.process_time()
                        distance_matrix_points_klein = distance_matrix(points_klein)
                        njGraph, nj_networkx = neighbor_joining(points_klein, distance_matrix_points_klein) 
                        del distance_matrix_points_klein ###
                        nj_adjacency_matrix, nj_nodes_ordered = adjacency_matrix(nj_networkx) 

                        nj_adjacency_matrix = torch.tensor(nj_adjacency_matrix)
                        poincare = klein_to_poincare(points_klein)
                        poincare = torch.tensor(poincare)

                        nj_steiners = train_steiner_embeddings(
                            nj_adjacency_matrix,
                            terminals_poincare=poincare,
                            num_epochs=10000,
                            lr=1)

                        combined = torch.cat([poincare, nj_steiners], dim=0)
                        combined_klein = poincare_to_klein(combined)
                        dist_matrix = distance_matrix(combined_klein, klein_distance)        
                        nj_length = (0.5*(dist_matrix*nj_adjacency_matrix)).sum()
                        nj_time = time.process_time() - start_time_nj

                        # Convert tensor to numpy value and clean up memory
                        nj_length_value = nj_length.detach().cpu().numpy().item()
                        
                        # Clean up tensors to free memory
                        del nj_adjacency_matrix, poincare, nj_steiners, combined, combined_klein, dist_matrix, nj_length
                        
                        # Force garbage collection to free GPU/CPU memory
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        gc.collect()
                                
                        #print(f"Success with seed {current_seed} for numPoints={numPoints}")
                        break

                    except Exception as e:
                        print(f"Error with seed {current_seed} for numPoints={numPoints}: {str(e)}")
                        if retry == max_retries - 1:
                            raise e 
                        continue

                avgTimeList.append(nj_time)
                ratioList.append(nj_length_value/MSTval)
                perImprovList.append((MSTval-nj_length_value)/MSTval)
                MSTvalList.append(MSTval)

                current_seed += 1

                i += 1


            avgTime = np.mean(avgTimeList)
            avgTime_std = np.std(avgTimeList)
            print(perImprovList)
            ratio = np.mean(ratioList)
            ratio_std = np.std(ratioList)
            perImprov = np.mean(perImprovList)*100
            perImprov_std = np.std(perImprovList)*100

            best_ratio = np.max(ratioList)
            best_perImprov = np.max(perImprovList)*100


            print(f"\r{numPoints}, {avgTime}+-{avgTime_std}, {best_perImprov}, {perImprov}+-{perImprov_std}, {ratio}+-{ratio_std}")

            numeric_results["mode"] += [mode]*numSamples
            numeric_results["numPoints"] += [numPoints]*numSamples
            numeric_results["avgTime"] += avgTimeList
            numeric_results["ratio"] += ratioList
            numeric_results["perImprov"] += perImprovList

    
            df = pd.DataFrame(numeric_results)


            df.to_csv(
                RESULT_PATH / f'results_{title}_{method}.tsv',
                sep="\t",
                index=False
                )

        print("-"*30)

# Ablation studies

In [ ]:
# RHS probability insertion ablation

# RHS
if True:
    numeric_results = defaultdict(lambda: list())
    samplingParam= {"cov": np.eye(2)*0.5}
    nIters = 3
    convDiff = 1e-1
    dist2Points = 1e-5
    numSamples = 10 
    title = "CenteredGauss_ProbaAblation"
    method="EXH"
    
    for mode in ["Klein_4+Simple"]:
        
       
        
        if mode == "Euclidean":
            sampleType = "EucGauss"
            fstSize = 4
            space = "Euclidean"
            precise=True
        else:
            sampleType = "WrappedGauss"
            space = "Klein"
            fstSize = int(mode[6])
            precise = mode[8:] == "Precise"
                
        for numPoints in [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]:

            for proba in [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]:
            
                nMaxExpansions=int(1*np.sqrt(numPoints))
                resultExp = syntheticExperiments(method=method,
                                                    numSamples=numSamples, 
                                                    numPoints=numPoints,
                                                    nIters=nIters,
                                                    fstSize=fstSize,
                                                    convDiff=convDiff,
                                                    dist2Points=dist2Points,
                                                    precise=precise, 
                                                    sampleType=sampleType,
                                                    samplingParam=samplingParam,
                                                    space=space,
                                                    nMaxExpansions=nMaxExpansions,
                                                    seed=10,
                                                    slack=2,
                                                    selection=("Prob", (proba, proba)),
                                                    threshold_improvement=0.0,
                                                    num_epochs=10000,
                                                    lr=1e-2,
                                                    early_stopping=True,
                                                    patience=100,
                                                    min_delta=1e-6)
    
    
    
    
                correctList, errVorList, avgTimeList, ratioList, perImprovList, avgFST3List, avgFST4List = resultExp
    
                correct = np.mean(correctList)*100
                avgTime = np.mean(avgTimeList)
                avgTime_std = np.std(avgTimeList)
                ratio = np.mean(ratioList)
                ratio_std = np.std(ratioList)
                perImprov = np.mean(perImprovList)*100
                perImprov_std = np.std(perImprovList)*100
                avgFST3 = np.mean(avgFST3List)
                avgFST4 = np.mean(avgFST4List)
    
                print(f"\r{mode}, {numPoints}, {proba}, {avgTime}+-{avgTime_std}, {perImprov}+-{perImprov_std}, {ratio}+-{ratio_std}")
    
                numeric_results["mode"] += [mode]*numSamples
                numeric_results["numPoints"] += [numPoints]*numSamples
                numeric_results["proba"] += [proba]*numSamples
                numeric_results["correct"] += correctList.tolist()
                numeric_results["avgTime"] += avgTimeList.tolist()
                #numeric_results["ratio"] += ratioList.tolist()
                numeric_results["perImprov"] += (100*perImprovList).tolist()
                numeric_results["ratio"] += ratioList.tolist()
                numeric_results["avgFST3"] += avgFST3List.tolist()
                numeric_results["avgFST4"] += avgFST4List.tolist()

           
            

                print("-"*30)

    
                df = pd.DataFrame(numeric_results)
            
            
                df.to_csv(
                    RESULT_PATH / f'results_{title}_{method}.tsv',
                    sep="\t",
                    index=False
                    )


In [ ]:
# RHS Expansion Schedule ablation
if True:
    numeric_results = defaultdict(lambda: list())
    sampleType = "PolyWrappedGauss" 
    space = "Klein"
    nIters = 3
    convDiff = 1e-1
    dist2Points = 1e-5
    numSamples = 3
    title = "Convergence"
    method="EXH"
    
    for mode in ["Klein_4+Simple"]:
        
        fstSize = int(mode[6])
        precise = mode[8:] == "Precise"
                
        for numPoints in [4,5,6,7,8,9,10, 20, 30, 40, 50, 60,70,80,90,100,200]:
            max_retries = 1000  # Maximum number of different seeds to try
            samplingParam= {"cov": np.eye(2)*0.01, "numPoly": numPoints, "t": 0.9}
            numPoints= numPoints*20
            for  expansion_mode in ["sqrt", "constant", "linear",]:

           
                nMaxExpansions=int(1*np.sqrt(numPoints))

                base_seed = 0
                resultExp = None

                for retry in range(max_retries):
                    current_seed = base_seed + retry
                    try:          
                        resultExp = syntheticExperiments(method=method,
                                                            numSamples=numSamples, 
                                                            numPoints=numPoints,
                                                            nIters=nIters,
                                                            fstSize=fstSize,
                                                            convDiff=convDiff,
                                                            dist2Points=dist2Points,
                                                            precise=precise, 
                                                            sampleType=sampleType,
                                                            samplingParam=samplingParam,
                                                            space=space,
                                                            nMaxExpansions=nMaxExpansions,
                                                            seed=current_seed,
                                                            slack=2,
                                                            selection=("Prob", (0.3, 0.6)),
                                                            threshold_improvement=0.0,
                                                            num_epochs=10000,
                                                                            # plot_graph="DT",

                                                            lr=1e-2,
                                                            early_stopping=True,
                                                            patience=100,
                                                            min_delta=1e-6,
                                                            expansion_mode=expansion_mode
                                                            )
                        
                        correctList, errVorList, avgTimeList, ratioList, perImprovList, avgFST3List, avgFST4List = resultExp
                        print(perImprovList)
                        while (min(perImprovList)<=0):
                            current_seed+=1
                            resultExp = syntheticExperiments(method=method,
                                                                numSamples=numSamples, 
                                                                numPoints=numPoints,
                                                                nIters=nIters,
                                                                fstSize=fstSize,
                                                                convDiff=convDiff,
                                                                dist2Points=dist2Points,
                                                                precise=precise, 
                                                                sampleType=sampleType,
                                                                samplingParam=samplingParam,
                                                                space=space,
                                                                nMaxExpansions=nMaxExpansions,
                                                                seed=current_seed,
                                                                slack=2,
                                                                                # plot_graph="DT",

                                                                selection=("Prob", (0.3, 0.6)),
                                                                threshold_improvement=0.0,
                                                                num_epochs=10000,
                                                                lr=1e-2,
                                                                early_stopping=True,
                                                                patience=100,
                                                                min_delta=1e-6,
                                                                expansion_mode=expansion_mode)        
                            correctList, errVorList, avgTimeList, ratioList, perImprovList, avgFST3List, avgFST4List = resultExp
                            print(perImprovList)


                        
                        if expansion_mode == "sqrt":
                            max_retries = retry+1
                
                        

                        print(f"Success with seed {current_seed} for numPoints={numPoints}")
                        break
                    except Exception as e:
                        print(f"Error with seed {current_seed} for numPoints={numPoints}: {str(e)}")
                        if retry == max_retries - 1:
                            raise e  
                        continue
                    
                 
                
                

                correct = np.mean(correctList)*100
                avgTime = np.mean(avgTimeList)
                avgTime_std = np.std(avgTimeList)
                ratio = np.mean(ratioList)
                ratio_std = np.std(ratioList)
                perImprov = np.mean(perImprovList)*100
                perImprov_std = np.std(perImprovList)*100
                avgFST3 = np.mean(avgFST3List)
                avgFST4 = np.mean(avgFST4List)

                best_ratio = np.max(ratioList)
                best_perImprov = np.max(perImprovList)*100
                
                print("hola", perImprovList*100, np.max(perImprovList)*100, np.mean(perImprovList)*100)



                print(f"\r{numPoints}, {expansion_mode}, {avgTime}+-{avgTime_std}, {best_perImprov}, {perImprov}+-{perImprov_std}, {ratio}+-{ratio_std}")

                numeric_results["mode"] += [mode]*numSamples
                numeric_results["numPoints"] += [numPoints]*numSamples
                numeric_results["correct"] += correctList.tolist()
                numeric_results["avgTime"] += avgTimeList.tolist()
                numeric_results["ratio"] += ratioList.tolist()
                numeric_results["perImprov"] += perImprovList.tolist()
                numeric_results["avgFST3"] += avgFST3List.tolist()
                numeric_results["avgFST4"] += avgFST4List.tolist()
                numeric_results["expansion_mode"] += [expansion_mode]*numSamples

        
                df = pd.DataFrame(numeric_results)


        print("-"*30)

# Visualization

In [ ]:
import re
from typing import List, Tuple, Optional, Sequence
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_from_latex_table(
    latex: str,
    methods_to_plot=None,
    show_theoretical=True,
    n_theoretical: int = 1000,
    std_scale: float = 1.0,
    style: str = "darkgrid",                      
    colors: Optional[dict] = None,                
    fs: float = 14,
    fill_alpha: float = 0.10,                    
    linewidth: float = 1.8,                       
    use_markers: bool = False,                    
    legend_loc: str = "upper center",
    bg_alpha: float = 0.5,                        
    xlim: Optional[Tuple[float,float]] = None,    
    ylim: Optional[Tuple[float,float]] = None,    
    xticks: Optional[Sequence[float]] = None,     
    xtick_labels: Optional[Sequence[str]] = None, 
    # --- log scale controls ---
    red_log_scale: bool = False,                  
    cpu_log_scale: bool = False                   
):
    """
    Parse the LaTeX table and plot RED/CPU with shaded std.
    """
    if methods_to_plot is None:
        methods_to_plot = ["NJ", "HS", "RHS"]
    else:
        valid_methods = {"NJ", "HS", "RHS"}
        methods_to_plot = [m for m in methods_to_plot if m in valid_methods]
        if not methods_to_plot:
            raise ValueError(f"No valid methods selected. Choose from {valid_methods}.")

    if colors is None:
        colors = {"NJ": "tab:green", "HS": "tab:red", "RHS": "tab:purple"}

    # --- helpers ---
    def parse_pm(cell: str) -> Tuple[float, float]:
        m = re.search(r"\$?\s*([+-]?\d+(?:\.\d+)?)\s*\\pm\s*([+-]?\d+(?:\.\d+)?)\s*\$?", cell)
        if not m:
            raise ValueError(f"Could not parse mean±std from cell: {cell!r}")
        return float(m.group(1)), float(m.group(2))

    def clean_row(line: str) -> List[str]:
        return [p.strip().rstrip("\\") for p in line.split("&")]

    # --- extract rows between first and second \midrule ---
    rows = []
    in_body = False
    for raw in latex.splitlines():
        line = raw.strip()
        if line.startswith(r"\midrule"):
            in_body = not in_body
            continue
        if in_body and re.match(r"^\d+", line):
            rows.append(line)
    if not rows:
        raise ValueError("No data rows found. Ensure LaTeX table has numeric rows between \\midrule lines.")

    data = {
        "P": [],
        "NJ_RED": [], "NJ_RED_STD": [], "NJ_CPU": [], "NJ_CPU_STD": [],
        "HS_RED": [], "HS_RED_STD": [], "HS_CPU": [], "HS_CPU_STD": [],
        "RHS_RED": [], "RHS_RED_STD": [], "RHS_CPU": [], "RHS_CPU_STD": [],
    }

    for line in rows:
        cells = clean_row(line)
        if len(cells) < 7:
            raise ValueError(f"Row has unexpected number of columns ({len(cells)}): {cells}")
        P = float(cells[0])  
        data["P"].append(P)

        nj_red_m, nj_red_s = parse_pm(cells[1]); nj_cpu_m, nj_cpu_s = parse_pm(cells[2])
        data["NJ_RED"] += [nj_red_m];  data["NJ_RED_STD"] += [nj_red_s]
        data["NJ_CPU"] += [nj_cpu_m];  data["NJ_CPU_STD"] += [nj_cpu_s]

        hs_red_m, hs_red_s = parse_pm(cells[3]); hs_cpu_m, hs_cpu_s = parse_pm(cells[4])
        data["HS_RED"] += [hs_red_m];  data["HS_RED_STD"] += [hs_red_s]
        data["HS_CPU"] += [hs_cpu_m];  data["HS_CPU_STD"] += [hs_cpu_s]

        rhs_red_m, rhs_red_s = parse_pm(cells[5]); rhs_cpu_m, rhs_cpu_s = parse_pm(cells[6])
        data["RHS_RED"] += [rhs_red_m]; data["RHS_RED_STD"] += [rhs_red_s]
        data["RHS_CPU"] += [rhs_cpu_m]; data["RHS_CPU_STD"] += [rhs_cpu_s]

    # Sort by P
    order = np.argsort(data["P"]).tolist()
    for k, v in data.items():
        data[k] = [v[i] for i in order]

    sns.set(style=style)  

    # Build some shared plot kwargs
    marker = "o" if use_markers else None

    # === RED plot ===
    fig_red, ax_red = plt.subplots(figsize=(7, 4.2))

    if show_theoretical:
        P_min, P_max = min(data["P"]), max(data["P"])
        P_smooth = np.linspace(P_min, P_max, 1000)
        theoretical = (1.0 - P_smooth / (2.0 * (P_smooth - 1.0))) * 100.0
        ax_red.plot(P_smooth, theoretical, color="black", linestyle="--", linewidth=linewidth,
                    label="Theoretical upper bound")
                    
    for method in methods_to_plot:
        y = np.array(data[f"{method}_RED"])
        ystd = np.array(data[f"{method}_RED_STD"]) / std_scale
        ax_red.fill_between(data["P"], y - ystd, y + ystd, color=colors[method], alpha=fill_alpha)
        # if (method == "NJ") and (show_theoretical == True):
        #     ax_red.plot(data["P"], y, color=colors[method], label=method, linestyle='--', dashes=(5, 10), linewidth=linewidth, marker=marker)

        # else:
        #     ax_red.plot(data["P"], y, color=colors[method], label=method, linewidth=linewidth, marker=marker)
        if (method != "RHS"):
            ax_red.plot(data["P"], y, color=colors[method], label=method, linestyle='--', dashes=(5, 5), linewidth=linewidth, marker=marker)
        else:
            ax_red.plot(data["P"], y, color=colors[method], label=method, linewidth=linewidth, marker=marker)


    ax_red.set_xlabel(r"$|P|$", fontsize=fs)
    ax_red.set_ylabel("RED (%)", fontsize=fs)
    #ax_red.set_title("RED vs |P|")
    if red_log_scale:
        ax_red.set_yscale('log')
    if legend_loc is not None:
        leg_red = ax_red.legend(loc=legend_loc, fontsize=16)
        leg_red.get_frame().set_facecolor("white")
    if xlim: ax_red.set_xlim(*xlim)
    if xticks is not None:
        ax_red.set_xticks(xticks)
        if xtick_labels is not None:
            ax_red.set_xticklabels(xtick_labels)
    ax_red.patch.set_alpha(bg_alpha)
    if ylim: ax_red.set_ylim(*ylim)
    ax_red.tick_params(axis='both', which='major', labelsize=16)
    fig_red.tight_layout()

    # === CPU plot ===
    fig_cpu, ax_cpu = plt.subplots(figsize=(7, 4.2))
    for method in methods_to_plot:
        y = np.array(data[f"{method}_CPU"])
        ystd = np.array(data[f"{method}_CPU_STD"])  
        ax_cpu.fill_between(data["P"], y - ystd, y + ystd, color=colors[method], alpha=fill_alpha)
        ax_cpu.plot(data["P"], y, color=colors[method], label=method, linewidth=linewidth, marker=marker)

    ax_cpu.set_xlabel(r"$|P|$", fontsize=fs)
    ax_cpu.set_ylabel("CPU (s)", fontsize=fs)
    ax_cpu.set_title("CPU vs |P|")
    if cpu_log_scale:
        ax_cpu.set_yscale('log')
    leg_cpu = ax_cpu.legend(loc=legend_loc)
    leg_cpu.get_frame().set_facecolor("white")
    if xlim: ax_cpu.set_xlim(*xlim)
    if xticks is not None:
        ax_cpu.set_xticks(xticks)
        if xtick_labels is not None:
            ax_cpu.set_xticklabels(xtick_labels)
    ax_cpu.patch.set_alpha(bg_alpha)
    ax_cpu.tick_params(axis='both', which='major', labelsize=17)
    fig_cpu.tight_layout()

    return fig_red, fig_cpu


In [ ]:
latex_table_exp1 = r"""\begin{table*}[ht!]
\centering
\resizebox{0.8\textwidth}{!}{
\begin{tabular}{lcccccccccc}
\toprule
\multicolumn{1}{c}{} & \multicolumn{2}{c}{NJ} & \multicolumn{2}{c}{HS} & \multicolumn{2}{c}{RHS (ours)} \\
\cmidrule(lr){2-3} 
\cmidrule(lr){4-5} 
\cmidrule(lr){6-7} 
$|P|$  & RED $(\uparrow)$ & CPU & RED $(\uparrow)$ & CPU & RED $(\uparrow)$ & CPU \\
\midrule
10    & $1.46 \pm 1.12$ & $5.50 \pm 0.03$ & $2.72 \pm 0.97$ & $0.03 \pm 0.01$ & $3.17 \pm 1.59$ & $4.60 \pm 3.40$ \\
20    & $-2.60 \pm 3.00$ & $5.68 \pm 0.04$ & $1.83 \pm 0.70$ & $0.06 \pm 0.01$ & $3.15 \pm 0.56$ & $9.87 \pm 7.00$ \\
30    & $-1.70 \pm 2.31$ & $5.86 \pm 0.03$ & $2.12 \pm 0.60$ & $0.10 \pm 0.02$ & $3.03 \pm 1.09$ & $12.65 \pm 6.94$ \\
40    & $-2.62 \pm 1.35$ & $5.95 \pm 0.04$ & $2.17 \pm 0.71$ & $0.13 \pm 0.02$ & $3.30 \pm 0.75$ & $17.59 \pm 6.97$ \\
50    & $-4.37 \pm 1.87$ & $6.14 \pm 0.04$ & $2.26 \pm 0.43$ & $0.17 \pm 0.02$ & $3.08 \pm 0.34$ & $26.21 \pm 8.67$ \\
60    & $-3.38 \pm 1.09$ & $6.29 \pm 0.03$ & $2.56 \pm 0.57$ & $0.20 \pm 0.02$ & $2.72 \pm 0.48$ & $26.95 \pm 8.40$ \\
70    & $-3.28 \pm 1.48$ & $6.51 \pm 0.04$ & $2.45 \pm 0.54$ & $0.22 \pm 0.01$ & $2.68 \pm 0.66$ & $31.09 \pm 11.84$ \\
80    & $-3.89 \pm 1.79$ & $6.77 \pm 0.03$ & $2.43 \pm 0.53$ & $0.26 \pm 0.02$ & $3.07 \pm 0.39$ & $38.55 \pm 8.45$ \\
90    & $-3.46 \pm 1.65$ & $7.01 \pm 0.04$ & $2.72 \pm 0.50$ & $0.30 \pm 0.02$ & $2.87 \pm 0.57$ & $66.07 \pm 30.15$ \\
100   & $-3.77 \pm 1.75$ & $7.24 \pm 0.03$ & $2.45 \pm 0.35$ & $0.33 \pm 0.02$ & $2.84 \pm 0.47$ & $70.31 \pm 14.92$ \\
150   & $-4.17 \pm 1.10$ & $8.46 \pm 0.03$ & $2.27 \pm 0.20$ & $0.48 \pm 0.02$ & $2.48 \pm 0.22$ & $139.87 \pm 49.03$ \\
200   & $-5.01 \pm 1.40$ & $11.09 \pm 0.06$ & $2.43 \pm 0.26$ & $0.66 \pm 0.04$ & $2.64 \pm 0.23$ & $341.42 \pm 96.02$ \\
\midrule
Avg.  &  &  &  &  &  &  \\
\bottomrule
\end{tabular}
}
\caption{Centered hyperbolic Gaussians $\mathcal{G}(0, 0.5)$. RED: reduction over MST (\%). CPU: total CPU time (sec.).}
\label{tab:exp1}
\end{table*}"""

In [ ]:
latex_table_exp2 = r"""\begin{table*}[ht!]
\centering
\resizebox{0.95\textwidth}{!}{
\begin{tabular}{lcccccc}
\toprule
\multicolumn{1}{c}{} & \multicolumn{2}{c}{NJ} & \multicolumn{2}{c}{HS} & \multicolumn{2}{c}{RHS} \\
\cmidrule(lr){2-3}
\cmidrule(lr){4-5}
\cmidrule(lr){6-7}
$|P|$ & RED & CPU & RED & CPU & RED & CPU \\
\midrule
10  & $40.49 \pm 0.03$ & $5.52 \pm 0.04$ & $17.75 \pm 4.57$ & $0.02 \pm 0.01$ & $28.33 \pm 18.54$ & $25.70 \pm 13.30$ \\
20  & $40.10 \pm 0.06$ & $5.73 \pm 0.05$ & $14.35 \pm 4.61$ & $0.05 \pm 0.02$ & $40.01 \pm 0.09$ & $61.94 \pm 16.77$ \\
30  & $39.93 \pm 0.08$ & $5.91 \pm 0.04$ & $14.74 \pm 5.75$ & $0.08 \pm 0.06$ & $39.84 \pm 0.13$ & $95.05 \pm 27.50$ \\
40  & $39.70 \pm 0.09$ & $6.06 \pm 0.03$ & $16.63 \pm 4.46$ & $0.09 \pm 0.03$ & $39.71 \pm 0.12$ & $119.86 \pm 53.15$ \\
50  & $39.53 \pm 0.06$ & $6.00 \pm 0.02$ & $17.88 \pm 5.29$ & $0.10 \pm 0.04$ & $39.52 \pm 0.05$ & $116.38 \pm 45.92$ \\
60  & $39.22 \pm 0.10$ & $6.37 \pm 0.04$ & $14.95 \pm 4.60$ & $0.11 \pm 0.02$ & $39.36 \pm 0.11$ & $87.99 \pm 9.33$ \\
70  & $39.01 \pm 0.16$ & $6.55 \pm 0.05$ & $14.42 \pm 4.27$ & $0.11 \pm 0.03$ & $39.26 \pm 0.05$ & $118.26 \pm 24.23$ \\
80  & $38.55 \pm 0.11$ & $6.77 \pm 0.04$ & $14.47 \pm 4.29$ & $0.12 \pm 0.03$ & $39.11 \pm 0.14$ & $111.56 \pm 12.72$ \\
90  & $38.18 \pm 0.18$ & $6.96 \pm 0.06$ & $15.85 \pm 3.70$ & $0.14 \pm 0.03$ & $39.03 \pm 0.08$ & $171.84 \pm 59.68$ \\
100 & $37.50 \pm 0.28$ & $7.26 \pm 0.07$ & $13.91 \pm 4.99$ & $0.13 \pm 0.04$ & $38.94 \pm 0.12$ & $183.57 \pm 60.30$ \\
150 & $24.02 \pm 1.24$ & $8.48 \pm 0.05$ & $13.22 \pm 6.21$ & $0.16 \pm 0.02$ & $38.47 \pm 0.10$ & $205.34 \pm 52.63$ \\
200 & $-8.08 \pm 1.09$ & $10.51 \pm 0.06$ & $13.62 \pm 4.43$ & $0.18 \pm 0.03$ & $38.17 \pm 0.17$ & $265.46 \pm 80.59$ \\
\midrule
Avg. &  &  &  &  &  &  \\
\bottomrule
\end{tabular}
}
\caption{Sampling $|P|$ points from a mixture of $10$ hyperbolic Gaussians $\mathcal{G}(\mu_{10, k}(1-10^{-10}), 0.1)$, $k \in \{1, \cdots, 10\}$. RED: reduction over MST (\%). CPU: total CPU time (sec.).}
\label{tab:exp2_new}
\end{table*}"""

In [ ]:
latex_table_exp3 = r"""\begin{table*}[ht!]
\centering
\resizebox{0.95\textwidth}{!}{
\begin{tabular}{lcccccc}
\toprule
\multicolumn{1}{c}{} & \multicolumn{2}{c}{NJ} & \multicolumn{2}{c}{HS} & \multicolumn{2}{c}{RHS (ours)} \\
\cmidrule(lr){2-3}
\cmidrule(lr){4-5}
\cmidrule(lr){6-7}
$|P|$ & RED $(\uparrow)$ & CPU & RED $(\uparrow)$ & CPU & RED $(\uparrow)$ & CPU \\
\midrule
4  & $31.31 \pm 0.07$ & $5.54 \pm 0.02$ & $31.34 \pm 0.08$ & $0.01 \pm 0.002$ & $31.39 \pm 0.11$ & $1.59 \pm 0.51$ \\
5  & $34.86 \pm 0.09$ & $5.58 \pm 0.02$ & $23.26 \pm 0.02$ & $0.01 \pm 0.003$ & $34.94 \pm 0.08$ & $10.26 \pm 1.44$ \\
6  & $36.95 \pm 0.10$ & $5.60 \pm 0.07$ & $18.28 \pm 0.12$ & $0.01 \pm 0.01$ & $37.01 \pm 0.10$ & $15.46 \pm 1.64$ \\
7  & $38.30 \pm 0.07$ & $5.59 \pm 0.02$ & $17.61 \pm 3.49$ & $0.01 \pm 0.005$ & $38.33 \pm 0.01$ & $13.09 \pm 4.26$ \\
8  & $39.26 \pm 0.03$ & $5.56 \pm 0.00$ & $10.81 \pm 6.10$ & $0.01 \pm 0.004$ & $39.28 \pm 0.02$ & $24.27 \pm 8.48$ \\
9  & $39.95 \pm 0.01$ & $5.59 \pm 0.03$ & $13.05 \pm 2.58$ & $0.01 \pm 0.01$ & $39.96 \pm 0.02$ & $52.79 \pm 18.82$ \\
10 & $40.48 \pm 0.01$ & $5.59 \pm 0.01$ & $11.60 \pm 2.38$ & $0.02 \pm 0.005$ & $40.51 \pm 0.02$ & $35.03 \pm 10.10$ \\
20 & $42.48 \pm 0.01$ & $5.76 \pm 0.04$ & $20.86 \pm 1.99$ & $0.17 \pm 0.03$ & $42.59 \pm 0.04$ & $92.29 \pm 14.17$ \\
30 & $42.93 \pm 0.004$ & $5.84 \pm 0.03$ & $21.04 \pm 2.51$ & $0.31 \pm 0.05$ & $43.03 \pm 0.11$ & $89.08 \pm 25.65$ \\
40 & $43.10 \pm 0.02$ & $6.01 \pm 0.03$ & $17.56 \pm 2.21$ & $0.33 \pm 0.02$ & $43.29 \pm 0.08$ & $159.69 \pm 41.75$ \\
50 & $43.16 \pm 0.01$ & $6.21 \pm 0.03$ & $20.07 \pm 1.17$ & $0.50 \pm 0.02$ & $43.26 \pm 0.06$ & $217.99 \pm 52.70$ \\
60 & $43.14 \pm 0.01$ & $6.40 \pm 0.03$ & $18.80 \pm 1.16$ & $0.58 \pm 0.03$ & $43.26 \pm 0.11$ & $214.50 \pm 64.05$ \\
70 & $43.12 \pm 0.003$ & $6.63 \pm 0.02$ & $18.84 \pm 1.01$ & $0.63 \pm 0.06$ & $43.39 \pm 0.04$ & $287.02 \pm 29.79$ \\
80 & $43.09 \pm 0.01$ & $6.81 \pm 0.04$ & $17.09 \pm 1.30$ & $0.75 \pm 0.06$ & $43.09 \pm 0.21$ & $239.36 \pm 76.02$ \\
90 & $43.05 \pm 0.01$ & $7.03 \pm 0.01$ & $18.65 \pm 0.75$ & $0.83 \pm 0.04$ & $43.15 \pm 0.13$ & $304.83 \pm 106.17$ \\
100 & $43.02 \pm 0.009$ & $7.20 \pm 0.03$ & $18.00 \pm 0.77$ & $0.80 \pm 0.01$ & $42.99 \pm 0.11$ & $310.98 \pm 54.61$ \\
\midrule
Avg. &  &  &  &  &  &  \\
\bottomrule
\end{tabular}
}
\caption{Mixture of hyperbolic Gaussians $\mathcal{G}(\mu_{10, k}(1-10^{-10}), 0.1)$, $k \in \{1, \cdots, 10\}$ towards the boundary of the hyperbolic disk where only one point is sampled for each $k$. RED: reduction over MST (\%). CPU: total CPU time (sec.).}
\label{tab:exp3_results}
\end{table*}"""


In [ ]:
fig1, fig2 = plot_from_latex_table(
    latex_table_exp1,
    methods_to_plot=["RHS", "HS", "NJ"],
    show_theoretical=False,          
    n_theoretical=1000,
    std_scale=1.0,                   
    style="darkgrid",                
    colors={"NJ":"#66800B","HS":"#AF3029","RHS":"#D0A215"},
    fs=18,
    fill_alpha=0.10,                 
    linewidth=2.0,
    use_markers=False,               
    legend_loc="center right",
    bg_alpha=0.5,                    
    xlim=(10, 200),                
    red_log_scale=False, 
    cpu_log_scale=True  
)

In [ ]:
fig1.savefig("Exp1_RED_plot_expscale.pdf", format="pdf", dpi=300, bbox_inches="tight")
fig1.savefig("Exp1_RED_plot_expscale.png", format="png", dpi=300, bbox_inches="tight")

In [ ]:
fig1, fig2 = plot_from_latex_table(
    latex_table_exp2,
    methods_to_plot=["RHS","NJ","HS"],
    show_theoretical=False,          
    n_theoretical=1000,
    std_scale=1.0,                   
    style="darkgrid",                
    colors={"NJ":"#66800B","HS":"#AF3029","RHS":"#D0A215"},
    fs=18,
    fill_alpha=0.10,                
    linewidth=2.0,
    use_markers=False,               
    legend_loc=None,
    bg_alpha=0.5,                    
    xlim=(10, 200),                 
)

In [ ]:
fig1.savefig("Exp2_RED_plot.pdf", format="pdf", dpi=300, bbox_inches="tight")
fig1.savefig("Exp2_RED_plot.png", format="png", dpi=300, bbox_inches="tight")

In [ ]:
fig1, fig2 = plot_from_latex_table(
    latex_table_exp3,
    methods_to_plot=["RHS", "NJ","HS"],
    show_theoretical=True,           
    n_theoretical=1000,
    std_scale=1.0,                   
    style="darkgrid",                
    colors={"NJ":"#66800B","HS":"#AF3029","RHS":"#D0A215"},
    fill_alpha=0.10,                 
    linewidth=2.0,
    fs=18,
    use_markers=False,               
    legend_loc="center right",
    bg_alpha=0.5,                   
    xlim=(3.5, 100),                
)

In [ ]:
fig1.savefig("Exp3_RED_plot.pdf", format="pdf", dpi=300, bbox_inches="tight")
fig1.savefig("Exp3_RED_plot.png", format="png", dpi=300, bbox_inches="tight")

In [ ]:
# Corresponding data visualization

numClusters = 10 #50 #100
n = numClusters
model = "Klein"
cov = np.eye(2)*0.1 

for i, t in enumerate([1-1e-10]):
    fig = plt.figure(figsize=(5,5), dpi=300)
    ax = fig.add_subplot(111, aspect='equal') 


    ax.set_xlim([-1.1,1.1])
    ax.set_ylim([-1.1,1.1])
    
    
    means = meansPolygon(n=numClusters, t=t)
    gaussian_samples = sampleExperiments("PolyWrappedGauss", n, samplingParam={"cov": cov, "t":t, "numPoly": numClusters})
    ax.scatter(gaussian_samples[:, 0], gaussian_samples[:, 1],
                c='tab:blue', s=50, alpha=0.7)
    
    circ = plt.Circle((0, 0), radius=1, edgecolor='grey', facecolor='None', linewidth=1, alpha=1)
    ax.add_patch(circ)
    
    ax.axis("off")
    
    plt.tight_layout()

    fig.savefig(FIG_PATH / f"aistatsexp3_{numClusters}.png")

    plt.show()